# PcamVLM — QLoRA Fine-Tune of MedGemma 1.5 4B on PCam + NCT-CRC-HE-100K

**Goal.** Fine-tune Google's [MedGemma 1.5 4B](https://arxiv.org/abs/2604.05081) (a SigLIP + Gemma3 medical VLM) to classify histopathology patches.

**Dataset.** 20,000 PCam (PatchCamelyon, CC0-1.0) + 10,000 NCT-CRC-HE-100K (CC BY 4.0) samples drawn from the public HuggingFace and Zenodo mirrors. Both auto-download on first run; falls back to a deterministic synthetic generator if no network is available.

**Model.** `google/medgemma-1.5-4b-it` — released Jan 13 2026, trained on CAMELYON, TCGA, and a broad medical mixture including whole-slide histopathology patches.

**Method.** QLoRA (4-bit nf4 + LoRA r=16 on Gemma3 attention + MLP). Trains ~30M of 3.9B parameters; fits a single Colab T4 (16 GB).

---

### Before running
1. `Runtime -> Change runtime type -> T4 GPU`
2. Accept the MedGemma license on HuggingFace: https://huggingface.co/google/medgemma-1.5-4b-it
3. Add your HuggingFace write token: `Runtime -> Secrets -> HF_TOKEN`



## Phase 1 — Install Dependencies


In [ ]:
# Install all required packages.
# transformers  - load model and processor (Gemma3 support >= 4.50)
# peft          - LoRA adapter
# bitsandbytes  - 4-bit nf4 quantization
# datasets      - load PCam from HuggingFace Hub
# accelerate    - distributed training utilities (needed by Trainer)
# trl           - SFTTrainer for QLoRA SFT
!pip install -q "transformers>=4.50" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.45" "datasets>=3.5" "trl>=0.12" pillow tqdm


In [ ]:
# Verify GPU + base libs
!nvidia-smi

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Log in to HuggingFace (MedGemma is a gated model).
# Colab Secrets is the recommended path; falls back to $HF_TOKEN otherwise.
import os, sys
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError("HF_TOKEN not set. Add it to Colab Secrets or os.environ.")
os.environ['HF_TOKEN'] = hf_token

from huggingface_hub import login
login(token=hf_token)
print("Logged in to HuggingFace")


## Phase 2 — Load PCam + NCT-CRC Datasets


PCam: 96x96 H&E patches of sentinel lymph node sections (binary tumor/normal).
NCT-CRC-HE-100K: 224x224 H&E patches from colorectal cancer histology (9 classes).
Both auto-download on first run; synthetic fallback if the network is unavailable.


In [ ]:
# Add project src/ to sys.path so we can import the package without pip install.
import sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up from cwd in case the notebook is run from a subdirectory
for d in [ROOT] + list(ROOT.parents):
    if (d / "src").is_dir():
        ROOT = d
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import data as D
print('PCam source:    ', D.PCAM_HF_ID)
print('NCT-CRC source: ', D.NCT_CRC_ZENODO_URL)
print('Cache dir:      ', D.DATA_CACHE)


In [ ]:
# Build a combined train + val split. Sizes auto-fall back to synthetic if the
# real datasets are unreachable.
TRAIN_SIZE = 20000
VAL_SIZE   = 2000

spec = D.DatasetSpec(task='mixed', train_size=TRAIN_SIZE, val_size=VAL_SIZE)
train_rows, val_rows = D.load_combined(spec, seed=42)
print(f'train rows: {len(train_rows)} (pcam={sum(1 for r in train_rows if r["task"]=="pcam")}, '
      f'nct_crc={sum(1 for r in train_rows if r["task"]=="nct_crc")})')
print(f'val rows:   {len(val_rows)}')

print('\nFirst train sample:')
print(f'  task:  {train_rows[0]["task"]}')
print(f'  label: {train_rows[0]["label"]}')
print(f'  image: {train_rows[0]["image"].size}')


In [ ]:
# Visualize a few samples side by side
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i, ax in enumerate(axes.flat):
    r = train_rows[i]
    ax.imshow(r['image'])
    name = 'no' if r['label']==0 else 'yes' if r['task']=='pcam' else __import__('src.core', fromlist=['NCT_CRC_LABEL_NAMES']).NCT_CRC_LABEL_NAMES.get(r['label'], str(r['label']))
    ax.set_title(f"{r['task']}: {name}", fontsize=9)
    ax.axis('off')
plt.suptitle('PcamVLM training samples (first 12)', fontsize=12)
plt.tight_layout()
plt.show()


## Phase 3 — Load MedGemma 1.5 4B in 4-bit


In [ ]:
import torch
from src import model as M

cfg = M.ModelConfig(
    base_model_id='google/medgemma-1.5-4b-it',
    torch_dtype='bfloat16',
    use_qlora=True,
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    min_pixels=224*224, max_pixels=896*896, image_size=224,
)

print('Loading processor...')
processor = M.load_processor(cfg)

print('Loading base model in 4-bit nf4 + bf16 compute...')
model, lora_config = M.load_model_and_adapter(cfg)
M.print_trainable_parameters(model)

print(f'\nModel loaded. dtype: {next(model.parameters()).dtype}')
if torch.cuda.is_available():
    print(f'VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')


In [ ]:
# Quick architecture overview
print('=== Top-level modules ===')
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f'  {name:30s}  {params/1e6:8.1f}M params')

total = sum(p.numel() for p in model.parameters())
print(f'\n  {"TOTAL":30s}  {total/1e6:8.1f}M params')


## Phase 4 — Training (TRL SFTTrainer + QLoRA)


In [ ]:
# Convert our list-of-dict rows to a HuggingFace Dataset of {messages: [...]}.
from datasets import Dataset
from src import core as C

def row_to_messages(row):
    q = C.PCAM_QUESTION if row['task']=='pcam' else C.nct_crc_question(C.NCT_CRC_LABEL_CHOICES)
    a = C.dataset_answer_for_label(row['task'], row['label'])
    return D.make_messages(row, q, a)

train_ds = Dataset.from_dict({'messages': [row_to_messages(r) for r in train_rows]})
val_ds   = Dataset.from_dict({'messages': [row_to_messages(r) for r in val_rows]})
print('train_ds:', train_ds)
print('val_ds:  ', val_ds)


In [ ]:
from trl import SFTConfig, SFTTrainer

sft_cfg = SFTConfig(
    output_dir='./pcam-medgemma-checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    optim='adamw_torch_fused',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    remove_unused_columns=False,
    max_length=2048,
    packing=False,
    seed=42,
)

print('Training config set.')
eff_bs = sft_cfg.per_device_train_batch_size * sft_cfg.gradient_accumulation_steps
print(f'Effective batch size: {eff_bs}')
print(f'Total train steps:    {len(train_ds) // eff_bs}')


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
)

print('Starting training...')
print('Watch the loss column - it should decrease each epoch.\n')

train_result = trainer.train()

print('\n=== Training Complete ===')
print(f'Total steps: {train_result.global_step}')
print(f'Train loss:  {train_result.training_loss:.4f}')


In [ ]:
# Plot training loss curve
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_losses = [(l['step'], l['loss']) for l in logs if 'loss' in l and 'eval_loss' not in l]
eval_losses  = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

if train_losses:
    steps_tr, losses_tr = zip(*train_losses)
    plt.figure(figsize=(10, 4))
    plt.plot(steps_tr, losses_tr, label='Train loss', color='steelblue')
    if eval_losses:
        steps_ev, losses_ev = zip(*eval_losses)
        plt.plot(steps_ev, losses_ev, label='Val loss', color='tomato', marker='o')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('MedGemma 1.5 4B QLoRA Fine-tuning on PCam + NCT-CRC')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Phase 5 — Save Adapter + Processor


In [ ]:
from src import persist as P
SAVE_DIR = './models/pcam-medgemma-lora'
metadata = P.TrainingMetadata(
    base_model=cfg.base_model_id,
    dataset=f'pcam+nct_crc (TRAIN_SIZE={TRAIN_SIZE}, VAL_SIZE={VAL_SIZE})',
    train_size=len(train_ds),
    val_size=len(val_ds),
    epochs=sft_cfg.num_train_epochs,
    lora_r=lora_config.r,
    lora_alpha=lora_config.lora_alpha,
    lora_dropout=lora_config.lora_dropout,
    learning_rate=sft_cfg.learning_rate,
    per_device_batch_size=sft_cfg.per_device_train_batch_size,
    gradient_accumulation_steps=sft_cfg.gradient_accumulation_steps,
    max_pixels=cfg.max_pixels,
    min_pixels=cfg.min_pixels,
    final_train_loss=float(train_result.training_loss),
    timestamp=__import__('datetime').datetime.utcnow().isoformat() + 'Z',
)
out = P.save_adapter(model, processor, SAVE_DIR, metadata=metadata)
print(f'Saved to {out}')
import os
for f in sorted(os.listdir(SAVE_DIR)):
    p = os.path.join(SAVE_DIR, f)
    if os.path.isfile(p):
        print(f'  {f}: {os.path.getsize(p)/1e6:.2f} MB')


In [ ]:
# Option B: push to HuggingFace Hub (optional).
# Replace YOUR_HF_USERNAME with your actual username.
YOUR_HF_USERNAME = 'YOUR_HF_USERNAME'   # <-- edit
if YOUR_HF_USERNAME != 'YOUR_HF_USERNAME':
    HF_REPO = f'{YOUR_HF_USERNAME}/pcam-medgemma-lora'
    model.push_to_hub(HF_REPO, private=False)
    processor.push_to_hub(HF_REPO, private=False)
    print(f'\nModel pushed to: https://huggingface.co/{HF_REPO}')
else:
    print('Set YOUR_HF_USERNAME to push to HuggingFace Hub.')


In [ ]:
# Option C: push the trained adapter to GitHub (for app.py deployment).
# The adapter is ~150-300 MB. GitHub's file limit is 100 MB, so use Git LFS if over 100 MB.
import os, subprocess

ADAPTER_DIR = './models/pcam-medgemma-lora'
GITHUB_REPO = 'https://github.com/bravo2024/PcamVLM.git'  # <-- update this
GITHUB_TOKEN = 'YOUR_GITHUB_TOKEN'  # <-- set to a token with repo write access

if GITHUB_TOKEN == 'YOUR_GITHUB_TOKEN':
    print('Set GITHUB_TOKEN to push to GitHub. Get one at https://github.com/settings/tokens')
else:
    total_mb = sum(os.path.getsize(os.path.join(r, f))/1e6 for r, _, fs in os.walk(ADAPTER_DIR) for f in fs)
    print(f'Adapter size: {total_mb:.1f} MB')

    !git config --global user.email 'colab@example.com'
    !git config --global user.name 'Colab Runner'

    if total_mb > 100:
        print('Adapter > 100 MB; enabling Git LFS...')
        !git lfs install
        !git lfs track 'models/pcam-medgemma-lora/**'
    else:
        print('Adapter < 100 MB; can commit directly.')

    !git add models/pcam-medgemma-lora/
    !git add .gitattributes
    !git commit -m 'Add MedGemma VLM adapter'
    !git push https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO.split('/')[-2]}/{GITHUB_REPO.split('/')[-1].replace('.git','')}.git master
    print('Pushed to GitHub.')


## Phase 6 — Evaluate on a Held-Out Test Set


In [ ]:
# Run inference on a held-out slice and compute accuracy / sens / spec / F1.
from src import infer, evaluate as E
from src.core import PCAM_LABEL_NAMES, NCT_CRC_LABEL_NAMES, NCT_CRC_LABEL_CHOICES
from tqdm import tqdm

TEST_N = 200  # keep small for notebook runtime
test_rows = val_rows[:TEST_N]

model.eval()
trues, preds = [], []

for r in tqdm(test_rows, desc='Evaluating'):
    img = r['image']
    if r['task'] == 'pcam':
        out = infer.predict_pcam(model, processor, img, use_logits=True)
    else:
        out = infer.predict_nct_crc(model, processor, img)
    trues.append(int(r['label']))
    preds.append(int(out['pred']))

# Combined report: split binary vs 9-class by task.
pcam_mask = [i for i, r in enumerate(test_rows) if r['task'] == 'pcam']
nct_mask  = [i for i, r in enumerate(test_rows) if r['task'] == 'nct_crc']
if pcam_mask:
    rep = E.evaluate_binary([trues[i] for i in pcam_mask], [preds[i] for i in pcam_mask])
    print(f'\n=== PCam (n={len(pcam_mask)}) ===')
    print(f'Accuracy: {rep.accuracy:.4f}')
    for k, v in rep.per_class.items():
        print(f'  class {k} ({v["name"]}): prec={v["precision"]:.3f} rec={v["recall"]:.3f} f1={v["f1"]:.3f}')
if nct_mask:
    rep = E.evaluate_multiclass([trues[i] for i in nct_mask], [preds[i] for i in nct_mask], NCT_CRC_LABEL_CHOICES)
    print(f'\n=== NCT-CRC (n={len(nct_mask)}) ===')
    print(f'Accuracy: {rep.accuracy:.4f}')
    macro_f1 = sum(v['f1'] for v in rep.per_class.values()) / len(rep.per_class)
    print(f'Macro F1: {macro_f1:.4f}')


## Phase 7 — Reload Adapter in a Fresh Session


In [ ]:
# Run this cell after restarting Colab to reload your fine-tuned adapter.
import torch
from transformers import AutoProcessor
from src.model import load_trained

BASE_MODEL_ID = 'google/medgemma-1.5-4b-it'
ADAPTER_PATH  = './pcam-medgemma-lora'      # local
# ADAPTER_PATH = 'YOUR_HF_USERNAME/pcam-medgemma-lora'   # from Hub

print('Loading base model...')
model_loaded = load_trained(BASE_MODEL_ID, ADAPTER_PATH, torch_dtype='bfloat16')
processor_loaded = AutoProcessor.from_pretrained(ADAPTER_PATH)

model_loaded.eval()
print('Model loaded and ready for inference!')


In [ ]:
# Run inference on a single PCam test patch
from src.infer import predict_pcam
from src.core import PCAM_QUESTION
import matplotlib.pyplot as plt

test_patch = D.load_pcam('test', 1, seed=42)[0]
img = test_patch['image']
true_label = test_patch['label']

out = predict_pcam(model_loaded, processor_loaded, img, use_logits=True)

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f"Pred: {out['pred_name']} | GT: {'yes' if true_label==1 else 'no'}\n"
          f"{'Correct' if out['pred']==true_label else 'Wrong'}", fontsize=11,
          color='green' if out['pred']==true_label else 'red')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Question: {PCAM_QUESTION}')
print(f'Pred:     {out["pred_name"]} (yes_prob={out.get("yes_prob", 0):.3f})')
print(f'True:     {"yes" if true_label==1 else "no"}')


In [ ]:
# Run inference on YOUR OWN uploaded image (Colab only).
try:
    from google.colab import files
    _IN_COLAB = True
except Exception:
    _IN_COLAB = False

from PIL import Image
import io

def _pad_to_square(img):
    w, h = img.size
    s = max(w, h)
    new = Image.new('RGB', (s, s), (0, 0, 0))
    new.paste(img, ((s - w) // 2, (s - h) // 2))
    return new

if not _IN_COLAB:
    print('Interactive upload only available in Google Colab.')
else:
    print('Upload a histopathology patch image...')
    uploaded = files.upload()
    for filename, data in uploaded.items():
        img = Image.open(io.BytesIO(data)).convert('RGB')
        img.thumbnail((224, 224), Image.LANCZOS)
        image = _pad_to_square(img).resize((224, 224), Image.LANCZOS)
        out = predict_pcam(model_loaded, processor_loaded, image, use_logits=True)
        plt.figure(figsize=(4, 4))
        plt.imshow(image)
        plt.title(f'Model answer: {out["pred_name"]} (P(yes)={out.get("yes_prob", 0):.2f})', fontsize=12)
        plt.axis('off')
        plt.show()
        print(f'File: {filename} -> {out["pred_name"]}')


## Summary


| Step | What you learned |
|---|---|
| 4-bit loading | Quantization - fit 3.9B medical VLM in ~7GB VRAM |
| LoRA | Train ~0.8% of weights, get most of the benefit |
| TRL SFTTrainer | QLoRA SFT with proper Gemma3 chat template |
| Multi-dataset | Train on PCam (binary) + NCT-CRC-HE-100K (9-class) |
| `save_pretrained` | Saves only the LoRA adapter (~30-80MB), not the 4B base |
| `PeftModel.from_pretrained` | Re-attaches adapter to frozen base model |
| Evaluation | Accuracy / sensitivity / specificity / F1 / confusion matrix |

### Next steps to improve accuracy
- Increase `TRAIN_SIZE` to 50k+ samples
- Try `r=32` in LoRA config (more capacity)
- Train for 2-3 epochs with a lower learning rate (5e-5)
- Add image augmentation (horizontal flip, color jitter)
- Eval on PathMMU, OmniMedVQA, ProbMed (the same benchmarks MedGemma was trained on)

### License & Citation
MedGemma 1.5 4B: [Health AI Developer Foundations terms](https://developers.google.com/health-ai-developer-foundations/terms).
PCam: CC0-1.0. NCT-CRC-HE-100K: CC BY 4.0 (Kather et al. 2018).
